# 01 - Carregamento e Preparação de Dados

Este notebook usa a configuração em YAML e os módulos do framework `model_supermarket` para carregar, normalizar e salvar a base de notas fiscais.

In [1]:
# IMPORTS
from pathlib import Path
import time
from src.core.config_loader import ConfigLoader
from src.core.tictoc import tictoc
from src.core.nfce_fetcher import run_nfce_fetch, process_base

# CONFIGURAÇÃO
tic_geral = time.time()
tic = time.time()
config = ConfigLoader("config/config.yaml")
qrcodes_path = Path(config.get('data.qrcodes_path'))
fetched_path = Path(config.get('data.fetched_data_path'))
mapping_path = Path(config.get('data.qrcode_mapping_path'))
raw_path = Path(config.get('data.raw_invoice_path'))
FETCH_REAL_DATA = config.get('data_fetching.enabled')
FETCH_REAL_DATA_TRAT = config.get('data_fetching.enabled_trat')
FETCH_DELAY = config.get('data_fetching.delay_between_requests')

## 1. Carregamento dos Dados

In [2]:
tic = time.time()

### 1.1 Busca no Site da Fazenda

**📦 `run_nfce_fetch`**

**📖 Descrição**

Função responsável por **buscar, extrair e estruturar dados de NFC-e a partir de QR Codes**, gerando uma base consolidada de itens de notas fiscais.

---

**⚙️ Parâmetros**

| Parâmetro       | Tipo    | Descrição                                                                |
| --------------- | ------- | ------------------------------------------------------------------------ |
| `fetch_enabled` | `bool`  | Se `True`, realiza a busca das notas. Se `False`, lê o CSV já existente. |
| `delay`         | `float` | Tempo de espera entre requisições.                                       |
| `qrcodes_path`  | `Path`  | Arquivo com os QR Codes (um por linha).                                  |
| `fetched_path`  | `Path`  | Caminho do CSV final gerado/lido.                                        |
| `mapping_path`  | `Path`  | Caminho do CSV de mapeamento da anonimização.                            |

---

**🔁 Comportamento**

**🔹 `fetch_enabled = False`**

* Lê o CSV existente (`fetched_path`)
* Retorna a base sem processamento

**🔹 `fetch_enabled = True`**

Executa as seguintes etapas:

* **Lê e remove QR Codes duplicados**
* **Consulta cada QR Code** (via API)
* **Extrai itens da nota**
* **Padroniza colunas** (schema final)
* **Converte e valida valores numéricos**
* **Enriquece com dados da nota**:

  * supermercado
  * CNPJ
  * data
  * totais
* **Anonimiza a chave da NFC-e** (SHA256)
* **Gera tabela de mapeamento (de_para)**
* **Consolida todas as notas em um único DataFrame**
* **Organiza colunas finais**
* **Salva em CSV**

---

**📤 Retorno**

`pd.DataFrame` com os itens das notas fiscais processadas.

### 1.2 Execução

In [3]:
df_fetched = run_nfce_fetch(
    FETCH_REAL_DATA,
    FETCH_DELAY,
    qrcodes_path,
    fetched_path,
    mapping_path,
)

Configuração de busca (nfceget): FETCH_REAL_DATA=True
ℹ QR Codes únicos: 141
[1/141] Buscando... ✅ Nota obtida
[2/141] Buscando... ✅ Nota obtida
[3/141] Buscando... ✅ Nota obtida
[4/141] Buscando... ✅ Nota obtida
[5/141] Buscando... ✅ Nota obtida
[6/141] Buscando... ✅ Nota obtida
[7/141] Buscando... ✅ Nota obtida
[8/141] Buscando... ✅ Nota obtida
[9/141] Buscando... ✅ Nota obtida
[10/141] Buscando... ✅ Nota obtida
[11/141] Buscando... ✅ Nota obtida
[12/141] Buscando... ✅ Nota obtida
[13/141] Buscando... ✅ Nota obtida
[14/141] Buscando... ✅ Nota obtida
[15/141] Buscando... ✅ Nota obtida
[16/141] Buscando... ✅ Nota obtida
[17/141] Buscando... ✅ Nota obtida
[18/141] Buscando... ✅ Nota obtida
[19/141] Buscando... ✅ Nota obtida
[20/141] Buscando... ✅ Nota obtida
[21/141] Buscando... ✅ Nota obtida
[22/141] Buscando... ✅ Nota obtida
[23/141] Buscando... ✅ Nota obtida
[24/141] Buscando... ✅ Nota obtida
[25/141] Buscando... ✅ Nota obtida
[26/141] Buscando... ✅ Nota obtida
[27/141] Buscando... ✅

### 1.3 Amostra da Base

In [4]:
print("Nº de Linhas e Colunas:", df_fetched.shape)
df_fetched.head()

Nº de Linhas e Colunas: (3164, 13)


,CHAVE_ANONIMIZADA,DATA,CNPJ,SUPERMERCADO,QTDE_TOTAL_NOTA,VALOR_TOTAL_NOTA,VALOR_TOTAL_TRIBUTOS,COD_PRODUTO,PRODUTO,UNIDADE,QTDE,VALOR_PRODUTO,VALOR_TOTAL_PRODUTO
0,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,278018,SALG ELMA CHIPS 40G,Un,1.0,3.69,3.69
1,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,278018,SALG ELMA CHIPS 40G,Un,1.0,3.69,3.69
2,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,264832,SALG ELMA CH 45G,Un,1.0,3.69,3.69
3,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,300337,AZEITE O LIVE 250ML,Un,1.0,21.90,21.90
4,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,12440,CERV SKOL 350ML,Un,12.0,2.88,34.56


In [5]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:02:54) [05/05/2026 20:52:16 - [05/05/2026 20:55:11]


## 2. Preparação dos Dados

In [6]:
tic = time.time()

### 2.1 Processo de Preparação

**📦 `process_base`**

**📖 Descrição**

Função responsável por **processar e enriquecer a base de NFC-e** ou **carregar uma versão já tratada**, dependendo do parâmetro `fetch_enabled`.

---

**⚙️ Parâmetros**

| Parâmetro       | Tipo           | Descrição                                                      |
| --------------- | -------------- | -------------------------------------------------------------- |
| `fetch_enabled` | `bool`         | Se `True`, processa os dados. Se `False`, apenas lê o `.xlsx`. |
| `df_fetched`    | `pd.DataFrame` | Base bruta gerada anteriormente.                               |
| `raw_path`      | `Path`         | Caminho do arquivo final `.xlsx`.                              |

---

**🔁 Comportamento**

**🔹 `fetch_enabled = False`**

* Lê o arquivo `.xlsx`
* Retorna a base já processada

**🔹 `fetch_enabled = True`**

Aplica as seguintes transformações:

* **Filtra supermercados** (Condor, Max, Assaí)
* **Padroniza nomes**
* **Cria features de tempo**:

  * `MES_ANO` (YYYYMM)
  * `PERIODO` (MADRUGADA, MANHA, TARDE, NOITE)
* **Resolve inconsistência de produtos** (mantém nome mais recente por código)
* **Classifica produtos** (`CAT_PRODUTO`)
* **Padroniza as unidades**
* **Organiza colunas finais**
* **Salva em `.xlsx`**

---

**📤 Retorno**

`pd.DataFrame` com a base tratada e pronta para análise.

### 2.2 Execução

In [7]:
df_trat = process_base(
    FETCH_REAL_DATA_TRAT,
    df_fetched,
    raw_path,
)

Configuração de busca (nfceget): FETCH_REAL_DATA=True
✅ CNPJ filtrados: 3
✅ Supermercados normalizados: 3
✅ Datas convertidas, meses/anos únicos: 43
✅ Períodos classificados: 3
✅ Produtos padronizados: 997
✅ Categorias de produtos classificadas: 19
✅ Unidades padronizadas: 10
✅ DataFrame final com colunas ordenadas
✅ DataFrame exportado para Excel (XLSX): data\notas_fiscais_supermercado.xlsx


### 2.3 Amostra da Base

In [8]:
print("Nº de Linhas e Colunas:", df_trat.shape)
df_trat.head()

Nº de Linhas e Colunas: (3079, 16)


,CHAVE_ANONIMIZADA,DATA,MES_ANO,PERIODO,CNPJ,SUPERMERCADO,QTDE_TOTAL_NOTA,VALOR_TOTAL_NOTA,VALOR_TOTAL_TRIBUTOS,COD_PRODUTO,CAT_PRODUTO,PRODUTO,UNIDADE,QTDE,VALOR_PRODUTO,VALOR_TOTAL_PRODUTO
1131,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,940360001,CONGELADO,TEKITOS 300G TRAD,UNIDADE,1.0,8.99,8.99
1135,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,11112050001,BEBIDA NAO ALCOOLICA,SUCO SUPER 3LT UVA,UNIDADE,1.0,17.90,17.90
1136,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,184710001,BEBIDA NAO ALCOOLICA,GATORADE 500ML UVA,GARRAFA,1.0,3.99,3.99
1137,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,11516920001,CEREAL,NESCAU ACTIV-GO 370G,UNIDADE,1.0,6.29,6.29
1138,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,202206,NOITE,06.057.223/0367-96,ASSAI,82,679.79,76.32,8290001,BEBIDA NAO ALCOOLICA,CHA LEAO NAT 40G,UNIDADE,1.0,3.49,3.49


In [9]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:02) [05/05/2026 20:55:11 - [05/05/2026 20:55:13]


## 3. Tempo Decorrido

In [10]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")


Tempo decorrido no notebook: (00:02:57) [05/05/2026 20:52:16 - [05/05/2026 20:55:13]
